In [ ]:
# Import library:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
# kiểm tra file
os.listdir()

In [ ]:
# Load file vào pandas
sales = pd.read_csv('sales.csv')
promotions = pd.read_csv('promotions.csv')
products = pd.read_csv('products.csv')
geography = pd.read_csv('geography.csv')
returns = pd.read_csv('returns.csv')
web_traffic = pd.read_csv('web_traffic.csv')
inventory = pd.read_csv('inventory.csv')
reviews = pd.read_csv('reviews.csv')
customers = pd.read_csv('customers.csv')
payments = pd.read_csv('payments.csv')
shipments = pd.read_csv('shipments.csv')
order_items = pd.read_csv('order_items.csv')
orders = pd.read_csv('orders.csv')
sample_submission = pd.read_csv('sample_submission.csv')

In [ ]:
# File Dictionary
dfs = {
    'sales': sales,
    'promotions': promotions,
    'products': products,
    'geography': geography,
    'returns': returns,
    'web_traffic': web_traffic,
    'inventory': inventory,
    'reviews': reviews,
    'customers': customers,
    'payments': payments,
    'shipments': shipments,
    'order_items': order_items,
    'orders': orders,
    'sample_submission': sample_submission
}

In [ ]:
# Check shape các file:
for name, df in dfs.items():
    print(f'{name}: {df.shape}')

In [ ]:
# Check giá trị đầu
for name, df in dfs.items():
    print(f'\n===== {name.upper()} =====')
    display(df.head())

In [ ]:
import sys
if 'google.colab' in sys.modules:
  %pip install ydata-profiling
from ydata_profiling import ProfileReport

In [ ]:
# brief EDA:

for name, df in dfs.items():
    if name != 'sample_submission':
        print(f"Generating ProfileReport for {name}...")
        profile = ProfileReport(df, title=f"{name.capitalize()} Report")
        profile.to_file(f"{name}_report.html")
        print(f"Report for {name} saved to {name}_report.html")

In [ ]:
# ktra Column name:
for name, df in dfs.items():
    print(f'\n===== {name.upper()} =====')
    print(df.columns.tolist())

In [ ]:
# Check thông tin từng bảng
for name, df in dfs.items():
    print(f'\n===== {name.upper()} =====')
    print(df.info())

In [ ]:
# Check missing value
for name, df in dfs.items():
    print(f'\n===== {name.upper()} =====')
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing) == 0:
        print('No missing values')
    else:
        print(missing)

In [ ]:
# Check dupl
for name, df in dfs.items():
    dup_count = df.duplicated().sum()
    print(f'{name}: duplicated rows = {dup_count}')

In [ ]:
# Basic Statistic:
for name, df in dfs.items():
    print(f'\n===== {name.upper()} =====')
    display(df.describe(include=[np.number]))

In [ ]:
# Check thống kê các object/categorical
for name, df in dfs.items():
    print(f'\n===== {name.upper()} =====')
    display(df.describe(include=['object']))

In [ ]:
# Check date time:
for name, df in dfs.items():
    possible_date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    print(f'{name}: {possible_date_cols}')

In [ ]:
# Quick capture:
audit_rows = []

for name, df in dfs.items():
    audit_rows.append({
        'table_name': name,
        'n_rows': df.shape[0],
        'n_cols': df.shape[1],
        'total_missing': int(df.isnull().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum()),
        'possible_date_cols': [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

In [ ]:
# check kĩ lại bảng sales
display(sales.head())
display(sales.info())
display(sales.isnull().sum())
display(sales.describe(include='all'))

In [ ]:
# check lại để bản submit đều ổn
display(sample_submission.head())
display(sample_submission.info())
display(sample_submission.isnull().sum())
display(sample_submission.describe(include='all'))

In [ ]:
# parse lại date
for name, df in dfs.items():
    date_cols = [col for col in df.columns if 'date' in col or 'time' in col]
    for col in date_cols:
        dfs[name][col] = pd.to_datetime(dfs[name][col], errors='coerce')

In [ ]:
for name, df in dfs.items():
    date_cols = [col for col in df.columns if 'date' in col or 'time' in col]
    if date_cols:
        print(f'\n{name}:')
        print(df[date_cols].dtypes)

In [ ]:
# Xóa duplicate:
for name in dfs:
    before = dfs[name].shape[0]
    dfs[name] = dfs[name].drop_duplicates()
    after = dfs[name].shape[0]
    if before != after:
        print(f'{name}: removed {before - after} duplicate rows')

In [ ]:
# Check lại missing sau khi parse date:
for name, df in dfs.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print(f'\n===== {name.upper()} =====')
    if len(missing) == 0:
        print('No missing values')
    else:
        print(missing)

In [ ]:
# Chuẩn hóa tên cột:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )
    return df

for name in dfs:
    dfs[name] = clean_column_names(dfs[name])

sales = dfs['sales']
promotions = dfs['promotions']
products = dfs['products']
geography = dfs['geography']
returns = dfs['returns']
web_traffic = dfs['web_traffic']
inventory = dfs['inventory']
reviews = dfs['reviews']
customers = dfs['customers']
payments = dfs['payments']
shipments = dfs['shipments']
order_items = dfs['order_items']
orders = dfs['orders']
sample_submission = dfs['sample_submission']

In [ ]:
display(promotions.head())
display(promotions.info())
display(promotions.isnull().sum())

In [ ]:
display(promotions.head())
display(promotions.info())
display(promotions.isnull().sum().sort_values(ascending=False))

In [ ]:
# clean bảng promotion:
promotions['applicable_category'] = promotions['applicable_category'].fillna('all_categories')
dfs['promotions'] = promotions
# giải thích: phần applicable này null có nghĩa là áp dụng cho tất cả category

In [ ]:
promotions.isnull().sum()

In [ ]:
# Clean bảng order items
order_items['promo_id'] = order_items['promo_id'].fillna('no_promo')
order_items['promo_id_2'] = order_items['promo_id_2'].fillna('no_second_promo')
dfs['order_items'] = order_items

In [ ]:
order_items.isnull().sum().sort_values(ascending=False)

In [ ]:
# Check logic giá trị trong order items
print("quantity < 0:", (order_items['quantity'] < 0).sum())
print("unit_price < 0:", (order_items['unit_price'] < 0).sum())
print("discount_amount < 0:", (order_items['discount_amount'] < 0).sum())

In [ ]:
# check logic giá trị trong sales
print("Revenue < 0:", (sales['revenue'] < 0).sum())
print("COGS < 0:", (sales['cogs'] < 0).sum())
print("COGS > Revenue:", (sales['cogs'] > sales['revenue']).sum())

In [ ]:
# check range thời gian trong promotion:
print("start_date > end_date:", (promotions['start_date'] > promotions['end_date']).sum())

In [ ]:
# check lại audit:
audit_rows = []

for name, df in dfs.items():
    audit_rows.append({
        'table_name': name,
        'n_rows': df.shape[0],
        'n_cols': df.shape[1],
        'total_missing': int(df.isnull().sum().sum()),
        'duplicate_rows': int(df.duplicated().sum())
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

In [ ]:
# Tạo data set với lõi từ
core_ts_df = sales[['date', 'revenue', 'cogs']].copy()
core_ts_df = core_ts_df.sort_values('date').reset_index(drop=True)

display(core_ts_df.head())
print(core_ts_df.shape)

In [ ]:
# Các bảng dùng cho exogenuous
exog_candidates = {
    'promotions': promotions.copy(),
    'web_traffic': web_traffic.copy(),
    'inventory': inventory.copy(),
    'orders': orders.copy(),
    'order_items': order_items.copy()
}

for name, df in exog_candidates.items():
    print(f'{name}: {df.shape}')

In [ ]:
# check lại thời gian trong các bảng exogenuous
for name, df in exog_candidates.items():
    date_cols = [col for col in df.columns if 'date' in col or 'time' in col]
    print(f'{name}: {date_cols}')

In [ ]:
print("Core dataset for forecasting:", core_ts_df.columns.tolist())
print("Exogenous candidate tables:", list(exog_candidates.keys()))

Chuẩn hóa core data set về time series

In [ ]:
# check lại data và sort thời gian
core_ts_df['date'] = pd.to_datetime(core_ts_df['date'], errors='coerce')
core_ts_df = core_ts_df.sort_values('date').reset_index(drop=True)

display(core_ts_df.head())
display(core_ts_df.dtypes)

In [ ]:
print("Min date:", core_ts_df['date'].min())
print("Max date:", core_ts_df['date'].max())
print("Total rows:", len(core_ts_df))
print("Unique dates:", core_ts_df['date'].nunique())

In [ ]:
ts_daily = core_ts_df[['date', 'revenue', 'cogs']].copy()

In [ ]:
# set index
ts_daily = ts_daily.set_index('date')

In [ ]:
# Gắn tần suất theo ngày
ts_daily = ts_daily.asfreq('D')

In [ ]:
# working data frame:
ts_model_df = ts_daily.copy()

In [ ]:
# quick check
display(ts_model_df.head())
display(ts_model_df.tail())
print(ts_model_df.shape)

# EDA cho time series trước khi modeling

Bước này chỉ tập trung nhìn 4 cái quan trọng nhất:

- xu hướng dài hạn

- seasonality theo tuần

- seasonality theo tháng / năm

- mối quan hệ giữa revenue và cogs

In [ ]:
plot_df = ts_model_df.reset_index().copy()
# Đưa date từ index về lại thành cột để vẽ biểu đồ và groupby dễ hơn.

In [ ]:
# Vẽ chuỗi thời gian của revenue:
plt.figure(figsize=(16, 5))
plt.plot(plot_df['date'], plot_df['revenue'])
plt.title('Daily Revenue Over Time')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.show()
# Biểu đồ này để check nhìn xu hướng tổng thể của doanh thu theo thời gian và xem có pattern mùa vụ rõ không.

In [ ]:
# Vẽ chuỗi thời gian của COGS:
plt.figure(figsize=(16, 5))
plt.plot(plot_df['date'], plot_df['cogs'])
plt.title('Daily COGS Over Time')
plt.xlabel('Date')
plt.ylabel('COGS')
plt.show()
# Biểu đồ này giúp check xem giá vốn có đi cùng xu hướng với doanh thu hay không.

* Dùng moving average để làm mượt dữ liệu*

In [ ]:
plot_df['revenue_ma7'] = plot_df['revenue'].rolling(7).mean()
plot_df['revenue_ma30'] = plot_df['revenue'].rolling(30).mean()

plt.figure(figsize=(16, 5))
plt.plot(plot_df['date'], plot_df['revenue'], alpha=0.4, label='Revenue')
plt.plot(plot_df['date'], plot_df['revenue_ma7'], label='7-day MA')
plt.plot(plot_df['date'], plot_df['revenue_ma30'], label='30-day MA')
plt.title('Revenue with 7-day and 30-day Moving Average')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.show()
# làm mượt chuỗi để nhìn rõ xu hướng ngắn hạn và dài hạn hơn, đặc biệt useful cho daily data.

In [ ]:
# Tạo feature ngày trong tuần và xem seasonality tuần
plot_df['day_of_week'] = plot_df['date'].dt.day_name()

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
revenue_dow = plot_df.groupby('day_of_week')['revenue'].mean().reindex(dow_order)
cogs_dow = plot_df.groupby('day_of_week')['cogs'].mean().reindex(dow_order)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(revenue_dow.index, revenue_dow.values, marker='o')
plt.title('Average Revenue by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Revenue')
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(cogs_dow.index, cogs_dow.values, marker='o')
plt.title('Average COGS by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average COGS')
plt.xticks(rotation=45)
plt.show()

Có trend theo tuần


In [ ]:
# Tạo feature tháng để nhìn pattern theo năm
plot_df['month'] = plot_df['date'].dt.month

revenue_month = plot_df.groupby('month')['revenue'].mean()
cogs_month = plot_df.groupby('month')['cogs'].mean()

In [ ]:
# plot trung bình theo tháng
plt.figure(figsize=(10, 4))
plt.plot(revenue_month.index, revenue_month.values, marker='o')
plt.title('Average Revenue by Month')
plt.xlabel('Month')
plt.ylabel('Average Revenue')
plt.xticks(range(1, 13))
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(cogs_month.index, cogs_month.values, marker='o')
plt.title('Average COGS by Month')
plt.xlabel('Month')
plt.ylabel('Average COGS')
plt.xticks(range(1, 13))
plt.show()

*Hai biểu đồ này giúp check xem có annual seasonality không, từ đó cân nhắc Fourier terms hoặc seasonal features theo năm*

In [ ]:
# Check tương quan giữa Revenue và COGS
plt.figure(figsize=(6, 6))
plt.scatter(plot_df['revenue'], plot_df['cogs'], alpha=0.3)
plt.title('Revenue vs COGS')
plt.xlabel('Revenue')
plt.ylabel('COGS')
plt.show()

print("Correlation:", plot_df[['revenue', 'cogs']].corr().iloc[0, 1])

In [ ]:
# margin để hiểu business dynamics
plot_df['gross_margin'] = plot_df['revenue'] - plot_df['cogs']

plt.figure(figsize=(16, 5))
plt.plot(plot_df['date'], plot_df['gross_margin'])
plt.title('Daily Gross Margin Over Time')
plt.xlabel('Date')
plt.ylabel('Gross Margin')
plt.show()

*Margin giúp bạn nhìn thêm chất lượng doanh thu theo thời gian, đôi khi rất hữu ích để giải thích behavior của revenue và cogs.*


# Quick summary:
weekly seasonality rõ

annual/monthly seasonality rõ

correlation rất mạnh giữa Revenue và COGS

# Baseline model:

In [ ]:
# Tạo train/validation set:
forecast_horizon = len(sample_submission) # dùng 548 ngày cuối để làm horizon của sample submission

train_df = ts_model_df.iloc[:-forecast_horizon].copy()
valid_df = ts_model_df.iloc[-forecast_horizon:].copy()

In [ ]:
# đưa date về cộtđể xử lý feature ngày trong tuần
train_base = train_df.reset_index().copy()
valid_base = valid_df.reset_index().copy()

In [ ]:
# Tạo feature day of week
train_base['day_of_week'] = train_base['date'].dt.day_name()
valid_base['day_of_week'] = valid_base['date'].dt.day_name()

In [ ]:
# Tính baseline trung bình theo ngày trong tuần từ tập train:
dow_baseline = train_base.groupby('day_of_week')[['revenue', 'cogs']].mean().reset_index()
display(dow_baseline)

In [ ]:
# Tạo dự báo baseline cho validation set :
valid_pred = valid_base[['date', 'day_of_week', 'revenue', 'cogs']].merge(
    dow_baseline,
    on='day_of_week',
    how='left',
    suffixes=('_actual', '_pred')
)

display(valid_pred.head())

In [ ]:
# hàm tính metric evaluate
def evaluate_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

In [ ]:
# đánh giá baseline của revenue:
revenue_mae, revenue_rmse = evaluate_forecast(
    valid_pred['revenue_actual'],
    valid_pred['revenue_pred']
)

print("Revenue baseline MAE:", revenue_mae)
print("Revenue baseline RMSE:", revenue_rmse)

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(valid_pred['date'], valid_pred['revenue_actual'], label='Actual Revenue')
plt.plot(valid_pred['date'], valid_pred['revenue_pred'], label='Baseline Revenue Pred')
plt.title('Baseline Forecast vs Actual - Revenue')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.show()

In [ ]:
# đánh giá baseline cho COGS:
cogs_mae, cogs_rmse = evaluate_forecast(
    valid_pred['cogs_actual'],
    valid_pred['cogs_pred']
)

print("COGS baseline MAE:", cogs_mae)
print("COGS baseline RMSE:", cogs_rmse)

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(valid_pred['date'], valid_pred['cogs_actual'], label='Actual COGS')
plt.plot(valid_pred['date'], valid_pred['cogs_pred'], label='Baseline COGS Pred')
plt.title('Baseline Forecast vs Actual - COGS')
plt.xlabel('Date')
plt.ylabel('COGS')
plt.legend()
plt.show()

In [ ]:
# lưu kết quả:
baseline_results = {
    'revenue_mae': revenue_mae,
    'revenue_rmse': revenue_rmse,
    'cogs_mae': cogs_mae,
    'cogs_rmse': cogs_rmse
}

baseline_results

# Tạo exogenous features cho SARIMAX:

In [ ]:
# Tạo 2 bảng nguồn: một bảng lịch sử để train model và một bảng future dates để tạo exog cho giai đoạn forecast.
sarimax_base_df = ts_model_df.reset_index().copy()
future_df = sample_submission[['date']].copy()

In [ ]:
# Chuẩn hóa kiểu dữ liệu của cột date
sarimax_base_df['date'] = pd.to_datetime(sarimax_base_df['date'])
future_df['date'] = pd.to_datetime(future_df['date'])

#Hàm sinh calendar features + Fourier terms

In [ ]:
def make_date_exog(df, date_col='date', origin_date=None, fourier_order=3):
    exog = df[[date_col]].copy()

    exog['day_of_week'] = exog[date_col].dt.dayofweek
    exog['month'] = exog[date_col].dt.month
    exog['quarter'] = exog[date_col].dt.quarter

    exog['is_weekend'] = (exog[date_col].dt.dayofweek >= 5).astype(int)
    exog['is_month_start'] = exog[date_col].dt.is_month_start.astype(int)
    exog['is_month_end'] = exog[date_col].dt.is_month_end.astype(int)
    exog['is_quarter_start'] = exog[date_col].dt.is_quarter_start.astype(int)
    exog['is_quarter_end'] = exog[date_col].dt.is_quarter_end.astype(int)

    exog['dow_sin'] = np.sin(2 * np.pi * exog['day_of_week'] / 7)
    exog['dow_cos'] = np.cos(2 * np.pi * exog['day_of_week'] / 7)

    exog['month_sin'] = np.sin(2 * np.pi * exog['month'] / 12)
    exog['month_cos'] = np.cos(2 * np.pi * exog['month'] / 12)

    if origin_date is None:
        origin_date = exog[date_col].min()

    t = (exog[date_col] - origin_date).dt.days

    for k in range(1, fourier_order + 1):
        exog[f'fourier_sin_{k}'] = np.sin(2 * np.pi * k * t / 365.25)
        exog[f'fourier_cos_{k}'] = np.cos(2 * np.pi * k * t / 365.25)

    return exog

In [ ]:
# Chọn mốc thời gian bắt đầu để tính fourier liên tục:
origin_date = sarimax_base_df['date'].min()

In [ ]:
# Tạo exogenous features cho dữ liệu lịch sử
exog_train_full = make_date_exog(
    sarimax_base_df,
    date_col='date',
    origin_date=origin_date,
    fourier_order=3
)

In [ ]:
# Tạo exogenous features cho future horizon
exog_future = make_date_exog(
    future_df,
    date_col='date',
    origin_date=origin_date,
    fourier_order=3
)

In [ ]:
# Bỏ cột date, chỉ giữ ma trận exog số
X_train_full = exog_train_full.drop(columns=['date']).copy()
X_future = exog_future.drop(columns=['date']).copy()
# SARIMAX cần ma trận exogenous dạng số, nên ở đây bỏ cột ngày và giữ lại toàn bộ feature số.

In [ ]:
# Đặt lại index để khớp với time series chính:
X_train_full.index = ts_model_df.index
X_future.index = future_df['date']

In [ ]:
# Tạo target riếng cho revenue và COGS:
y_revenue = ts_model_df['revenue'].copy()
y_cogs = ts_model_df['cogs'].copy()

In [ ]:
# LOG transform để chuẩn bị modelling:
y_revenue_log = np.log1p(y_revenue)
y_cogs_log = np.log1p(y_cogs)

In [ ]:
# Quick check output
display(X_train_full.head())
display(X_future.head())

print("X_train_full shape:", X_train_full.shape)
print("X_future shape:", X_future.shape)
print("y_revenue_log shape:", y_revenue_log.shape)
print("y_cogs_log shape:", y_cogs_log.shape)

# Split dữ liệu và fit SARIMAX

In [ ]:
# Tạo validation horizon và split dữ liệu lịch sử

validation_horizon = len(sample_submission)

y_rev_train = y_revenue_log.iloc[:-validation_horizon]
y_rev_valid = y_revenue_log.iloc[-validation_horizon:]

y_cogs_train = y_cogs_log.iloc[:-validation_horizon]
y_cogs_valid = y_cogs_log.iloc[-validation_horizon:]

X_train = X_train_full.iloc[:-validation_horizon].copy()
X_valid = X_train_full.iloc[-validation_horizon:].copy()

In [ ]:
# Fit SARIMAX đầu tiên cho Revenue
revenue_model = SARIMAX(
    y_rev_train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

revenue_result = revenue_model.fit(disp=False)

In [ ]:
# Forecast Revenue trên validation:
revenue_valid_pred_log = revenue_result.forecast(
    steps=validation_horizon,
    exog=X_valid
)

revenue_valid_pred = np.expm1(revenue_valid_pred_log)
revenue_valid_actual = np.expm1(y_rev_valid)

In [ ]:
# Đánh giá Revenue SARIMAX
revenue_sarimax_mae, revenue_sarimax_rmse = evaluate_forecast(
    revenue_valid_actual,
    revenue_valid_pred
)

print("Revenue SARIMAX MAE:", revenue_sarimax_mae)
print("Revenue SARIMAX RMSE:", revenue_sarimax_rmse)

In [ ]:
# actual vs forecast cho Revenue:
plt.figure(figsize=(16, 5))
plt.plot(revenue_valid_actual.index, revenue_valid_actual.values, label='Actual Revenue')
plt.plot(revenue_valid_actual.index, revenue_valid_pred.values, label='SARIMAX Revenue Pred')
plt.title('SARIMAX Forecast vs Actual - Revenue')
plt.xlabel('Date')
plt.ylabel('Revenue')
plt.legend()
plt.show()

In [ ]:
# Fit SARIMAX đầu tiên cho COGS:
cogs_model = SARIMAX(
    y_cogs_train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

cogs_result = cogs_model.fit(disp=False)

In [ ]:
# Forecast COGS trên validation:
cogs_valid_pred_log = cogs_result.forecast(
    steps=validation_horizon,
    exog=X_valid
)

cogs_valid_pred = np.expm1(cogs_valid_pred_log)
cogs_valid_actual = np.expm1(y_cogs_valid)

In [ ]:
# Đánh giá COGS SARIMAX:
cogs_sarimax_mae, cogs_sarimax_rmse = evaluate_forecast(
    cogs_valid_actual,
    cogs_valid_pred
)

print("COGS SARIMAX MAE:", cogs_sarimax_mae)
print("COGS SARIMAX RMSE:", cogs_sarimax_rmse)

In [ ]:
# actual vs forecast cho COGS:
plt.figure(figsize=(16, 5))
plt.plot(cogs_valid_actual.index, cogs_valid_actual.values, label='Actual COGS')
plt.plot(cogs_valid_actual.index, cogs_valid_pred.values, label='SARIMAX COGS Pred')
plt.title('SARIMAX Forecast vs Actual - COGS')
plt.xlabel('Date')
plt.ylabel('COGS')
plt.legend()
plt.show()

In [ ]:
# lưu kết quả:
sarimax_results_v1 = {
    'revenue_mae': revenue_sarimax_mae,
    'revenue_rmse': revenue_sarimax_rmse,
    'cogs_mae': cogs_sarimax_mae,
    'cogs_rmse': cogs_sarimax_rmse
}

sarimax_results_v1

In [ ]:
# So sánh nhanh:
print("=== Revenue ===")
print("Baseline MAE :", baseline_results['revenue_mae'])
print("SARIMAX  MAE :", sarimax_results_v1['revenue_mae'])
print("Baseline RMSE:", baseline_results['revenue_rmse'])
print("SARIMAX  RMSE:", sarimax_results_v1['revenue_rmse'])

print("\n=== COGS ===")
print("Baseline MAE :", baseline_results['cogs_mae'])
print("SARIMAX  MAE :", sarimax_results_v1['cogs_mae'])
print("Baseline RMSE:", baseline_results['cogs_rmse'])
print("SARIMAX  RMSE:", sarimax_results_v1['cogs_rmse'])

# Fine-tunning

In [ ]:
# Fine-tuning setup: tạo lại train/validation split for SARIMAX
validation_horizon = len(sample_submission)

y_rev_train = y_revenue_log.iloc[:-validation_horizon]
y_rev_valid = y_revenue_log.iloc[-validation_horizon:]

y_cogs_train = y_cogs_log.iloc[:-validation_horizon]
y_cogs_valid = y_cogs_log.iloc[-validation_horizon:]

X_train = X_train_full.iloc[:-validation_horizon].copy()
X_valid = X_train_full.iloc[-validation_horizon:].copy()

In [ ]:
# Fine-tuning search space for SARIMAX hyperparameters
tuning_orders = [
    (1, 1, 1),
    (2, 1, 1),
    (1, 1, 2),
    (2, 1, 2),
    (1, 0, 1)
]

tuning_seasonal_orders = [
    (1, 1, 1, 7),
    (1, 0, 1, 7),
    (0, 1, 1, 7),
    (1, 1, 0, 7)
]

In [ ]:
# Fine-tuning utility: run one SARIMAX hyperparameter configuration
def run_sarimax_tuning_experiment(y_train, y_valid, X_train, X_valid, order, seasonal_order):
    try:
        model = SARIMAX(
            y_train,
            exog=X_train,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        result = model.fit(disp=False)

        pred_log = result.forecast(steps=len(y_valid), exog=X_valid)
        pred = np.expm1(pred_log)
        actual = np.expm1(y_valid)

        mae, rmse = evaluate_forecast(actual, pred)

        return {
            'order': order,
            'seasonal_order': seasonal_order,
            'aic': result.aic,
            'mae': mae,
            'rmse': rmse,
            'status': 'ok'
        }

    except Exception as e:
        return {
            'order': order,
            'seasonal_order': seasonal_order,
            'aic': np.nan,
            'mae': np.nan,
            'rmse': np.nan,
            'status': f'fail: {str(e)[:120]}'
        }

In [ ]:
# Fine-tuning Revenue SARIMAX hyperparameters
revenue_tuning_results = []

for order in tuning_orders:
    for seasonal_order in tuning_seasonal_orders:
        print(f'Fine-tuning Revenue | order={order}, seasonal_order={seasonal_order}')

        res = run_sarimax_tuning_experiment(
            y_train=y_rev_train,
            y_valid=y_rev_valid,
            X_train=X_train,
            X_valid=X_valid,
            order=order,
            seasonal_order=seasonal_order
        )

        revenue_tuning_results.append(res)

In [ ]:
# Fine-tuning results summary for Revenue
revenue_tuning_df = pd.DataFrame(revenue_tuning_results)
revenue_tuning_df = revenue_tuning_df.sort_values(['rmse', 'mae'], ascending=True).reset_index(drop=True)

display(revenue_tuning_df)

In [ ]:
# Best fine-tuned SARIMAX configuration for Revenue
best_revenue_config = revenue_tuning_df.iloc[0].to_dict()
best_revenue_config

In [ ]:
# Fine-tuning COGS SARIMAX hyperparameters
cogs_tuning_results = []

for order in tuning_orders:
    for seasonal_order in tuning_seasonal_orders:
        print(f'Fine-tuning COGS | order={order}, seasonal_order={seasonal_order}')

        res = run_sarimax_tuning_experiment(
            y_train=y_cogs_train,
            y_valid=y_cogs_valid,
            X_train=X_train,
            X_valid=X_valid,
            order=order,
            seasonal_order=seasonal_order
        )

        cogs_tuning_results.append(res)

In [ ]:
# Fine-tuning COGS SARIMAX hyperparameters
cogs_tuning_results = []

for order in tuning_orders:
    for seasonal_order in tuning_seasonal_orders:
        print(f'Fine-tuning COGS | order={order}, seasonal_order={seasonal_order}')

        res = run_sarimax_tuning_experiment(
            y_train=y_cogs_train,
            y_valid=y_cogs_valid,
            X_train=X_train,
            X_valid=X_valid,
            order=order,
            seasonal_order=seasonal_order
        )

        cogs_tuning_results.append(res)

In [ ]:
# Fine-tuning results summary for COGS
cogs_tuning_df = pd.DataFrame(cogs_tuning_results)
cogs_tuning_df = cogs_tuning_df.sort_values(['rmse', 'mae'], ascending=True).reset_index(drop=True)

display(cogs_tuning_df)

In [ ]:
# Best fine-tuned SARIMAX configuration for COGS
best_cogs_config = cogs_tuning_df.iloc[0].to_dict()
best_cogs_config

In [ ]:
# Fine-tuning comparison: baseline vs initial SARIMAX vs tuned SARIMAX
print("=== REVENUE ===")
print("Baseline RMSE :", baseline_results['revenue_rmse'])
print("SARIMAX v1 RMSE:", sarimax_results_v1['revenue_rmse'])
print("Tuned SARIMAX RMSE:", best_revenue_config['rmse'])

print("\n=== COGS ===")
print("Baseline RMSE :", baseline_results['cogs_rmse'])
print("SARIMAX v1 RMSE:", sarimax_results_v1['cogs_rmse'])
print("Tuned SARIMAX RMSE:", best_cogs_config['rmse'])

In [ ]:
# Save best fine-tuned hyperparameters for final refit
best_finetuned_configs = {
    'revenue': {
        'order': best_revenue_config['order'],
        'seasonal_order': best_revenue_config['seasonal_order']
    },
    'cogs': {
        'order': best_cogs_config['order'],
        'seasonal_order': best_cogs_config['seasonal_order']
    }
}

best_finetuned_configs

# Refit best fine-tunning SARIMAX model


In [ ]:
# Refit setup: load best fine-tuned SARIMAX hyperparameters
best_revenue_order = best_finetuned_configs['revenue']['order']
best_revenue_seasonal = best_finetuned_configs['revenue']['seasonal_order']

best_cogs_order = best_finetuned_configs['cogs']['order']
best_cogs_seasonal = best_finetuned_configs['cogs']['seasonal_order']

print("Best Revenue config:", best_revenue_order, best_revenue_seasonal)
print("Best COGS config:", best_cogs_order, best_cogs_seasonal)

In [ ]:
# Refit best fine-tuned SARIMAX for Revenue on full historical data
from statsmodels.tsa.statespace.sarimax import SARIMAX

final_revenue_model = SARIMAX(
    y_revenue_log,
    exog=X_train_full,
    order=best_revenue_order,
    seasonal_order=best_revenue_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_revenue_result = final_revenue_model.fit(disp=False)

In [ ]:
# Refit best fine-tuned SARIMAX for COGS on full historical data
final_cogs_model = SARIMAX(
    y_cogs_log,
    exog=X_train_full,
    order=best_cogs_order,
    seasonal_order=best_cogs_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
)

final_cogs_result = final_cogs_model.fit(disp=False)

In [ ]:
# Forecast future horizon for Revenue
future_revenue_pred_log = final_revenue_result.forecast(
    steps=len(X_future),
    exog=X_future
)

In [ ]:
# Forecast future horizon for COGS
future_cogs_pred_log = final_cogs_result.forecast(
    steps=len(X_future),
    exog=X_future
)

In [ ]:
# Inverse transform predictions back to original scale
future_revenue_pred = np.expm1(future_revenue_pred_log)
future_cogs_pred = np.expm1(future_cogs_pred_log)

In [ ]:
# post-processing: clip negative predictions to zero
future_revenue_pred = np.clip(future_revenue_pred, 0, None)
future_cogs_pred = np.clip(future_cogs_pred, 0, None)

In [ ]:
# Create submission dataframe using sample submission format
submission_df = sample_submission.copy()
submission_df['revenue'] = future_revenue_pred.values
submission_df['cogs'] = future_cogs_pred.values

display(submission_df.head())
display(submission_df.tail())
print(submission_df.shape)

In [ ]:
# Sanity check for submission output
print("Revenue min prediction:", submission_df['revenue'].min())
print("Revenue max prediction:", submission_df['revenue'].max())
print("COGS min prediction:", submission_df['cogs'].min())
print("COGS max prediction:", submission_df['cogs'].max())
print("Any missing in submission:", submission_df.isnull().sum().sum())

In [ ]:
# Business sanity check: compare COGS and Revenue predictions
print("Number of days where COGS > Revenue:", (submission_df['cogs'] > submission_df['revenue']).sum())

In [ ]:
# visualization: inspect future forecast curves
plt.figure(figsize=(16, 5))
plt.plot(submission_df['date'], submission_df['revenue'], label='Forecast Revenue')
plt.plot(submission_df['date'], submission_df['cogs'], label='Forecast COGS')
plt.title('Future Forecast - Revenue and COGS')
plt.xlabel('Date')
plt.ylabel('Predicted Value')
plt.legend()
plt.show()

In [ ]:
# Match exact column names required by Kaggle sample submission
submission_df = submission_df.rename(columns={
    'date': 'Date',
    'revenue': 'Revenue',
    'cogs': 'COGS'
})

display(submission_df.head())
print(submission_df.columns.tolist())

In [ ]:
# Export corrected submission file
submission_df.to_csv('submission.csv', index=False)
print("Saved file: submission.csv")